# Import libs

In [1]:
import os
import json
from pathlib import Path
import numpy as np
import torch


libgomp: Invalid value for environment variable OMP_NUM_THREADS
/home/jovyan/dnalm/my_saved_conda_envs/gena/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
[10, 11, 12]
[0, 0, 0, 1, 1, 1, 2, 2,2]

[0, 0, 0, 1, 1, 1, 2, 2, 2]

# Configure model

In [3]:
os.chdir('/home/jovyan/dnalm/')

In [4]:
rmt_model_path = Path('/home/jovyan/dnalm/runs/annotation/bert_base_512_lastln_t2t_1000G_bs256_lr_1e-04_linear_fp16/model_2000000/rmt_seglen_512_len42000_maxnsegm_10000_msz_5_bptt-1_lr5e-05_AdamW_cosine_wd1e-04_p10_bs64_it50000/run_1/')

In [5]:
exp_config = json.load((rmt_model_path / 'config.json').open('r')) # it should be config.json that I sent you

In [6]:
for k in ['input_seq_len', 'model_cfg', 'model_cls', 'backbone_cls', 'input_size', 'num_mem_tokens', 'max_n_segments', 'tokenizer']:
    print(f'{k}: {exp_config[k]}')

input_seq_len: 42000
model_cfg: ./data/configs/L12-H768-A12-V32k-preln-lastln.json
model_cls: src.gena_lm.modeling_rmt:RMTEncoderForTokenClassification
backbone_cls: src.gena_lm.modeling_bert:BertForTokenClassification
input_size: 512
num_mem_tokens: 5
max_n_segments: 10000
tokenizer: ./data/tokenizers/t2t_1000h_multi_32k/


In [7]:
from transformers import AutoTokenizer, AutoConfig
tokenizer = AutoTokenizer.from_pretrained('./data/tokenizers/t2t_1000h_multi_32k/')

In [8]:
from src.gena_lm.modeling_bert import BertForLetterLevelTokenClassification, BertForTokenClassification

In [9]:
model_cfg = AutoConfig.from_pretrained('./data/configs/L12-H768-A12-V32k-preln-lastln.json') # here it soulbe config for backbone model, don't change it, you can change only path to it
# model_cfg.num_labels = 6
# model_cfg.problem_type = 'multi_label_classification'
model_cls = BertForLetterLevelTokenClassification
model = model_cls(config=model_cfg)

In [10]:
model_cfg_small = AutoConfig.from_pretrained('./data/configs/L12-H768-A12-V32k-preln-lastln-small.json') # here it soulbe config for backbone model, don't change it, you can change only path to it
model_cfg_small.num_labels = 6
model_cfg_small.problem_type = 'multi_label_classification'
model_cls_small = BertForTokenClassification
model_small = model_cls_small(config=model_cfg_small)

In [11]:
rmt_config = {
            'num_mem_tokens': exp_config['num_mem_tokens'],
            'max_n_segments': exp_config['max_n_segments'],
            'input_size': exp_config['input_size'],
            'bptt_depth': -1,
            'sum_loss': True,
            'tokenizer': tokenizer
        }
from src.gena_lm.modeling_rmt import RMTEncoderForLetterLevelTokenClassification
rmt_cls = RMTEncoderForLetterLevelTokenClassification
model_rmt = rmt_cls(model, model_small, **rmt_config)

In [12]:
# load pre-trained weights
ckpt = torch.load(str('/home/jovyan/dnalm/runs/annotation/bert_base_512_lastln_t2t_1000G_bs256_lr_1e-04_linear_fp16/model_2000000/rmt_seglen_512_len42000_maxnsegm_10000_msz_5_bptt-1_lr5e-05_AdamW_cosine_wd1e-04_p10_bs64_it50000/run_1/model_best/accelerate_state/pytorch_model.bin'), map_location='cpu')
missing_k, unexpected_k = model_rmt.load_state_dict(ckpt, strict=False)
print(f'missing: {missing_k}') # if no missing tensors - that is correct, otherwise - no!
print(f'unexpected_k: {unexpected_k}') # if no missing tensors - that is correct, otherwise - no!

missing: ['sub_model.bert.embeddings.position_ids', 'sub_model.bert.embeddings.word_embeddings.weight', 'sub_model.bert.embeddings.position_embeddings.weight', 'sub_model.bert.embeddings.token_type_embeddings.weight', 'sub_model.bert.embeddings.LayerNorm.weight', 'sub_model.bert.embeddings.LayerNorm.bias', 'sub_model.bert.encoder.layer.0.pre_attention_ln.weight', 'sub_model.bert.encoder.layer.0.pre_attention_ln.bias', 'sub_model.bert.encoder.layer.0.post_attention_ln.weight', 'sub_model.bert.encoder.layer.0.post_attention_ln.bias', 'sub_model.bert.encoder.layer.0.attention.self.query.weight', 'sub_model.bert.encoder.layer.0.attention.self.query.bias', 'sub_model.bert.encoder.layer.0.attention.self.key.weight', 'sub_model.bert.encoder.layer.0.attention.self.key.bias', 'sub_model.bert.encoder.layer.0.attention.self.value.weight', 'sub_model.bert.encoder.layer.0.attention.self.value.bias', 'sub_model.bert.encoder.layer.0.attention.output.dense.weight', 'sub_model.bert.encoder.layer.0.atte

In [12]:
model = model.eval()

# Evaluation example

In [13]:
seq = 'ATGC'*2900
seq2 = 'ATGC'*900
sequences = [seq, seq2]
input_features = tokenizer(sequences, return_tensors='pt', padding=True, truncation=True)

input_features['labels_mask'] = (input_features['input_ids'] > 5).long() # dumb realization

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [29]:
tokenizer.convert_ids_to_tokens(100)

'TTTTC'

In [14]:
input_features['labels_mask']

tensor([[0, 1, 1,  ..., 1, 1, 0],
        [0, 1, 1,  ..., 0, 0, 0]])

In [15]:
# # check the length of the sequence in tokens with no PADDING (but with SEP and CLS)
# no_padding_input_features = tokenizer([seq, seq2], return_tensors='pt', padding=True, truncation=True)['input_ids']
# no_padding_input_features.shape

In [16]:
input_features['input_ids'].shape

torch.Size([2, 1452])

In [17]:
input_features.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels_mask'])

In [18]:
input_features['labels'] = torch.randint(0, 6, (len(sequences), input_features['input_ids'].shape[1], 6))

In [19]:
input_features['letter_level_labels'] = torch.randint(0, 1, (len(sequences), len(seq), 6))

In [20]:
input_features['labels'].shape

torch.Size([2, 1452, 6])

In [21]:
input_features['letter_level_labels'].shape

torch.Size([2, 11600, 6])

In [22]:
input_features['embedding_repeater'] = torch.randint(2, 10, (len(sequences), len(seq),))
input_features['embedding_repeater'].shape

torch.Size([2, 11600])

In [23]:
input_features['letter_level_tokens'] = (torch.ones((len(sequences), len(seq))) + 20).long()
input_features['letter_level_tokens'].shape

torch.Size([2, 11600])

In [24]:
lllm = torch.tensor([[1] * len(seq), [1] * len(seq2) + [0] * (len(seq) - len(seq2))]).bool()
input_features['letter_level_labels_mask'] = lllm

In [25]:
input_features.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels_mask', 'labels', 'letter_level_labels', 'embedding_repeater', 'letter_level_tokens', 'letter_level_labels_mask'])

In [26]:
with torch.no_grad():
    out = model_rmt(**input_features)

google
torch.Size([1, 1450, 768])
torch.Size([1, 11600, 6])
google
torch.Size([1, 450, 768])
torch.Size([1, 11600, 6])


In [27]:
out.keys()

odict_keys(['loss', 'logits'])

In [28]:
out.loss

tensor(0.8631)

In [29]:
out.logits.shape

torch.Size([2, 11600, 6])

In [30]:
torch.nn.functional.pad(out.logits, (0, 0, 0, len(seq) - len(seq2))).shape

torch.Size([2, 19600, 6])

In [31]:
# you can handle it like this
out.logits[:, input_features['input_ids'][0] > 5, :].shape

IndexError: The shape of the mask [1452] at index 0 does not match the shape of the indexed tensor [2, 11600, 6] at index 1

In [ ]:
out.logits[0, 0, :] # use softmax to get probabilities